# Capítulo 4 · Descrevendo uma variável: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, uma coorte de 300 adultos internados com pneumonia adquirida na comunidade, pelas perguntas do autor; aparecem só as seções com cálculo (4.2, 4.4 a 4.15, 4.21, 4.22 e 4.23). O livro desenha as figuras; o caderno imprime os mesmos números. A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 19; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (16, 17 e 19), o código e o resultado. O código é Python 3 com pandas. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/cohort.csv`: 300 linhas, uma por paciente internado em 2025, e 11 colunas: `patient_id`, `admitted_on`, `sex`, `age_years`, `ward` (medicine, respiratory, geriatrics, intensive care), `severity` (low, moderate, high), `comorbidities`, `sbp_mmhg`, `spo2_pct`, `crp_mg_l` e `los_days` (permanência). `data/triage.csv` é uma cópia do registro de triagem do Capítulo 3, usada só pelo Exercício 17. Cada célula traz o número da seção do livro a que pertence.

### 4.2 Um primeiro olhar sobre os dados com o pandas

*Como o arquivo da coorte pode ser lido e inspecionado, e o que a biblioteca supõe sobre cada coluna?*

In [1]:
# 1. O caminho da biblioteca padrão: um dicionário reúne as permanências de cada enfermaria, e cada valor é convertido de texto para número inteiro
import csv

stays = {}
with open("data/cohort.csv", newline="") as file:
    for row in csv.DictReader(file):
        if row["ward"] not in stays:
            stays[row["ward"]] = []
        stays[row["ward"]].append(int(row["los_days"]))

# 2. Um laço calcula a média de cada lista
for ward in stays:
    days = stays[ward]
    print("Enfermaria", ward + ":", len(days), "pacientes, permanência média", round(sum(days) / len(days), 2), "dias")

Enfermaria medicine: 136 pacientes, permanência média 7.46 dias
Enfermaria geriatrics: 49 pacientes, permanência média 7.98 dias
Enfermaria respiratory: 88 pacientes, permanência média 6.97 dias
Enfermaria intensive care: 27 pacientes, permanência média 11.37 dias


In [2]:
# 3. O caminho do pandas: a tabela inteira em um objeto (um DataFrame), uma linha por paciente e uma coluna por variável
import pandas as pd

df = pd.read_csv("data/cohort.csv")
print("A coorte tem", df.shape[0], "linhas (uma por paciente) e", df.shape[1], "colunas.")
print("Tipo de armazenamento de los_days:", df.dtypes["los_days"], "; de severity:", df.dtypes["severity"])

# 4. Os três primeiros registros
print(df.head(3))

A coorte tem 300 linhas (uma por paciente) e 11 colunas.
Tipo de armazenamento de los_days: int64 ; de severity: str
  patient_id admitted_on sex  age_years         ward  severity  comorbidities  \
0       C001  2025-02-18   F         69     medicine       low              0   
1       C002  2025-03-28   M         73   geriatrics  moderate              0   
2       C003  2025-12-28   M         86  respiratory       low              3   

   sbp_mmhg  spo2_pct  crp_mg_l  los_days  
0       134        96       116         1  
1        93        93        69         8  
2        97        95        51         4  


### 4.4 Distribuição de frequências

*Com que frequência cada valor ocorre?*

In [3]:
# 1. Distribuição de frequências da gravidade, na ordem da escala: absoluta (F), relativa em % (Fr), acumulada (Fac) e relativa acumulada em % (Frac)
order = ["low", "moderate", "high"]
table = pd.DataFrame({"F": df["severity"].value_counts().reindex(order)})
table["Fr"] = (100 * table["F"] / len(df)).round(2)
table["Fac"] = table["F"].cumsum()
table["Frac"] = (100 * table["Fac"] / len(df)).round(2)
print(table)

# 2. Idades agrupadas em classes de dez anos, fechadas à esquerda: 30-39, 40-49 e assim por diante
limits = [30, 40, 50, 60, 70, 80, 90, 100]
age_class = pd.cut(df["age_years"], bins=limits, right=False)
print("Pacientes em cada classe de idade:", age_class.value_counts().sort_index().tolist())

            F     Fr  Fac    Frac
severity                         
low        97  32.33   97   32.33
moderate  129  43.00  226   75.33
high       74  24.67  300  100.00
Pacientes em cada classe de idade: [8, 16, 53, 77, 79, 50, 17]


### 4.5 Desenhando com o matplotlib

*Como uma variável é desenhada em Python?*

In [4]:
# 1. As alturas das barras do gráfico de gravidade são as contagens de cada classe, na ordem da escala (o livro as desenha)
counts = df["severity"].value_counts().reindex(order)
print("Alturas das barras:", counts.tolist())

Alturas das barras: [97, 129, 74]


### 4.6 Gráfico de barras

*Como as categorias devem ser mostradas em um gráfico de barras?*

In [5]:
# 1. As barras das enfermarias, da mais à menos frequente (o livro as desenha)
wards_count = df["ward"].value_counts()
print("Pacientes por enfermaria, da mais à menos frequente:", wards_count.tolist())

Pacientes por enfermaria, da mais à menos frequente: [136, 88, 49, 27]


### 4.7 Gráfico de Pareto

*Quais motivos respondem pela maior parte dos atrasos?*

In [6]:
# 1. Motivos de atraso, do mais ao menos frequente, e suas porcentagens acumuladas (o livro desenha o gráfico)
reasons = pd.Series({"radiograph": 31, "not suspected": 19,
                     "late assessment": 14, "drug unavailable": 9,
                     "venous access": 6, "other": 3})
cumulative = 100 * reasons.cumsum() / reasons.sum()
print("Porcentagens acumuladas:", cumulative.round(1).tolist())

Porcentagens acumuladas: [37.8, 61.0, 78.0, 89.0, 96.3, 100.0]


### 4.8 Histograma

*Que forma a permanência toma?*

In [7]:
# 1. Contagens da permanência em classes de cinco dias: 1-5, 6-10 e assim por diante até 66 (o livro desenha o histograma)
import numpy as np

counts, limits = np.histogram(df["los_days"], bins=range(1, 67, 5))
print("Contagens das seis primeiras classes:", counts[:6].tolist())

Contagens das seis primeiras classes: [148, 93, 32, 10, 5, 5]


### 4.9 Diagrama de ramo e folhas

*Os próprios valores podem ser mostrados mantendo a forma?*

In [8]:
# 1. Diagrama de ramo e folhas das 30 primeiras pressões sistólicas: o ramo é a dezena, a folha é a unidade
values = sorted(df["sbp_mmhg"].head(30))
stems = {}
for value in values:
    stem = value // 10
    if stem not in stems:
        stems[stem] = []
    stems[stem].append(value % 10)

# 2. Uma linha por ramo
for stem in sorted(stems):
    leaves = ""
    for leaf in stems[stem]:
        leaves = leaves + str(leaf) + " "
    print(str(stem).rjust(3), "|", leaves)

  9 | 3 7 
 10 | 5 
 11 | 1 7 
 12 | 0 3 3 4 5 6 6 7 8 9 
 13 | 3 3 4 6 7 9 
 14 | 0 0 4 6 7 
 15 | 0 5 9 
 16 | 1 


### 4.10 Média aritmética

*Qual é o centro das permanências, medido pela média aritmética?*

In [9]:
# 1. Permanência média: os dez primeiros pacientes (cálculo manual) e os 300 pacientes
print("Média das dez primeiras permanências:", df["los_days"].head(10).mean(), "dias; das 300 permanências:", round(df["los_days"].mean(), 2), "dias")

# 2. Por enfermaria: a média simples das médias das enfermarias contra a média ponderada (a média da coorte)
wards = df.groupby("ward")["los_days"].agg(["count", "mean"])
weighted = (wards["count"] * wards["mean"]).sum() / wards["count"].sum()
print("Média simples das médias das enfermarias:", round(wards["mean"].mean(), 2), "dias; média ponderada:", round(weighted, 2), "dias")

Média das dez primeiras permanências: 5.8 dias; das 300 permanências: 7.75 dias
Média simples das médias das enfermarias: 8.44 dias; média ponderada: 7.75 dias


### 4.11 Mediana

*Qual é a mediana da permanência, e ela resiste a valores incomuns?*

In [10]:
# 1. Mediana da permanência: os dez primeiros pacientes (cálculo manual) e os 300 pacientes
print("Mediana das dez primeiras permanências:", df["los_days"].head(10).median(), "dias; das 300 permanências:", df["los_days"].median(), "dias")

# 2. Um erro de digitação em uma cópia da coluna: 620 dias digitados no lugar de 62
typo = df["los_days"].replace(62, 620)
print("Com o erro de digitação: média", round(typo.mean(), 2), "dias; mediana", typo.median(), "dias")

# 3. Pacientes cujas permanências são menores que a média
print("Pacientes com permanência menor que a média:", (df["los_days"] < df["los_days"].mean()).sum())

Mediana das dez primeiras permanências: 5.5 dias; das 300 permanências: 6.0 dias
Com o erro de digitação: média 9.61 dias; mediana 6.0 dias
Pacientes com permanência menor que a média: 196


### 4.12 Gráfico de linhas

*Como as internações variaram ao longo dos doze meses?*

In [11]:
# 1. Internações em cada mês de 2025 e sua mediana (o livro desenha o gráfico de linhas)
month = pd.to_datetime(df["admitted_on"]).dt.month
monthly = month.value_counts().sort_index()
print("Internações por mês, de janeiro a dezembro:", monthly.tolist())
print("Mediana das doze contagens mensais:", monthly.median())

Internações por mês, de janeiro a dezembro: [39, 40, 32, 16, 7, 14, 8, 19, 13, 29, 33, 50]
Mediana das doze contagens mensais: 24.0


### 4.13 Moda

*Qual valor ocorre com mais frequência?*

In [12]:
# 1. A moda da enfermaria e da permanência (um método que devolve todos os valores empatados na maior frequência)
print("Enfermaria modal:", df["ward"].mode().tolist(), "; permanência modal:", df["los_days"].mode().tolist(), "dias")

# 2. A moda de uma variável medida com precisão contra a sua mediana
print("Pressão sistólica modal:", df["sbp_mmhg"].mode().tolist(), "mmHg; mediana:", df["sbp_mmhg"].median(), "mmHg")

Enfermaria modal: ['medicine'] ; permanência modal: [3] dias
Pressão sistólica modal: [133] mmHg; mediana: 126.0 mmHg


### 4.14 Quantis: quartis, decis e percentis

*Quais valores dividem as permanências em quartos e décimos?*

In [13]:
# 1. Quartis e percentil 90 das dez primeiras permanências (cálculo manual)
print("Dez primeiras permanências - quartil 1, quartil 3 e percentil 90:", df["los_days"].head(10).quantile([0.25, 0.75, 0.90]).tolist())

# 2. Os percentis 10, 25, 50, 75 e 90 das 300 permanências
print("300 permanências - percentis 10, 25, 50, 75 e 90:", df["los_days"].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).tolist())

Dez primeiras permanências - quartil 1, quartil 3 e percentil 90: [4.0, 7.75, 11.0]
300 permanências - percentis 10, 25, 50, 75 e 90: [2.0, 3.0, 6.0, 9.25, 14.0]


### 4.15 Boxplot e outliers

*Quais permanências são outliers?*

In [14]:
# 1. Quartis, intervalo interquartil e as duas cercas (1,5 e 3 vezes o intervalo interquartil acima do terceiro quartil)
los = df["los_days"]
q1 = los.quantile(0.25)
q3 = los.quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
extreme = q3 + 3 * iqr
print("Intervalo interquartil:", iqr, "dias; cerca inferior:", q1 - 1.5 * iqr, "dias; cerca superior:", upper, "dias; cerca extrema:", extreme, "dias")

# 2. O bigode superior e as permanências sinalizadas
print("Maior permanência dentro da cerca:", los[los <= upper].max(), "dias")
print("Permanências além da cerca superior:", (los > upper).sum(), "; além da cerca extrema:", (los > extreme).sum())

Intervalo interquartil: 6.25 dias; cerca inferior: -6.375 dias; cerca superior: 18.625 dias; cerca extrema: 28.0 dias
Maior permanência dentro da cerca: 18 dias
Permanências além da cerca superior: 23 ; além da cerca extrema: 8


### 4.21 Assimetria

*A distribuição é simétrica?*

In [15]:
# 1. Assimetria (ajustada para amostras) de três variáveis: positiva para cauda direita mais longa, negativa para cauda esquerda mais longa
for name in ["los_days", "spo2_pct", "sbp_mmhg"]:
    print("Assimetria de", name + ":", round(df[name].skew(), 2))

Assimetria de los_days: 3.16
Assimetria de spo2_pct: -1.42
Assimetria de sbp_mmhg: -0.06


### 4.22 Curtose

*Quão pesadas são as caudas?*

In [16]:
# 1. Curtose em excesso (ajustada para amostras) das mesmas três variáveis: zero para a distribuição normal
for name in ["los_days", "spo2_pct", "sbp_mmhg"]:
    print("Curtose em excesso de", name + ":", round(df[name].kurt(), 2))

Curtose em excesso de los_days: 14.23
Curtose em excesso de spo2_pct: 2.54
Curtose em excesso de sbp_mmhg: -0.24


### 4.23 Tabela 1: descrevendo a amostra

*Como a coorte inteira é descrita na primeira tabela do manuscrito?*

In [17]:
# 1. Valores não faltantes e resumo das colunas numéricas da Tabela 1
cols = ["age_years", "sex", "severity", "comorbidities", "sbp_mmhg", "spo2_pct", "crp_mg_l"]
print("Valores não faltantes de cada coluna:", df[cols].count().tolist())
print(df[cols].describe().round(1))

# 2. Contagens e porcentagens das categorias
sex_counts = df["sex"].value_counts().reindex(["F", "M"])
print("Sexo (F, M): contagens", sex_counts.tolist(), "; porcentagens", (100 * sex_counts / len(df)).round(1).tolist())
severity_counts = df["severity"].value_counts().reindex(["low", "moderate", "high"])
print("Gravidade (baixa, moderada, alta): contagens", severity_counts.tolist(), "; porcentagens", (100 * severity_counts / len(df)).round(1).tolist())

Valores não faltantes de cada coluna: [300, 300, 300, 300, 300, 300, 300]
       age_years  comorbidities  sbp_mmhg  spo2_pct  crp_mg_l
count      300.0          300.0     300.0     300.0     300.0
mean        68.5            2.0     126.1      95.2     108.7
std         13.8            1.4      18.8       3.5     112.0
min         31.0            0.0      73.0      80.0       8.0
25%         59.0            1.0     114.0      94.0      42.0
50%         69.0            2.0     126.0      96.0      73.5
75%         78.0            3.0     139.0      98.0     132.8
max         97.0            6.0     172.0     100.0     982.0
Sexo (F, M): contagens [134, 166] ; porcentagens [44.7, 55.3]
Gravidade (baixa, moderada, alta): contagens [97, 129, 74] ; porcentagens [32.3, 43.0, 24.7]


## Parte B. Soluções dos exercícios

### 4.25 Exercício 1. Destino de alta

| Destino | $F_i$ | $Fr_i$ (%) |
|---|---|---|
| Casa | 52 | 65,00 |
| Casa com apoio de enfermagem | 14 | 17,50 |
| Unidade de reabilitação | 9 | 11,25 |
| Instituição de longa permanência | 5 | 6,25 |
| Total | 80 | 100,00 |


A variável é nominal: os destinos não têm uma ordem imposta pelos dados.
As frequências absoluta e relativa fazem sentido. As colunas acumuladas não,
porque mudariam com a ordem arbitrária das linhas.

### 4.25 Exercício 2. Atendimentos no pronto-socorro

(a) As frequências absolutas 12, 11, 8, 5, 3, 1 resultam em frequências relativas de 30,0; 27,5;
20,0; 12,5; 7,5 e 2,5%, frequências acumuladas de 12, 23, 31, 36, 39, 40 e
porcentagens acumuladas de 30,0; 57,5; 77,5; 90,0; 97,5; 100. Dois ou mais atendimentos:
$8+5+3+1=17$ pacientes, $17/40=42{,}5\%$, o que é também $100-57{,}5$. (b) A média é
$(0\times12+1\times11+2\times8+3\times5+4\times3+6\times1)/40=60/40=1{,}5$ atendimento.

### 4.25 Exercício 3. Tempos porta-agulha

A expressão de Sturges dá $k=1+3{,}322\times\log_{10}30\approx5{,}9$, cerca de seis
classes; a amplitude é $118-18=100$ minutos, então $h\approx17$. Classes de 20 minutos
de largura a partir de 0 são mais fáceis de ler e resultam em seis classes:

| Minutos | $F_i$ | $Fr_i$ (%) | $Fac_i$ | $Frac_i$ (%) |
|---|---|---|---|---|
| $[0;20)$ | 1 | 3,33 | 1 | 3,33 |
| $[20;40)$ | 12 | 40,00 | 13 | 43,33 |
| $[40;60)$ | 10 | 33,33 | 23 | 76,67 |
| $[60;80)$ | 4 | 13,33 | 27 | 90,00 |
| $[80;100)$ | 2 | 6,67 | 29 | 96,67 |
| $[100;120)$ | 1 | 3,33 | 30 | 100,00 |


Menos de 60 minutos: 23 de 30 pacientes, 76,67%. Outras larguras arredondadas são aceitáveis
se os limites e a regra para as fronteiras forem declarados.

### 4.25 Exercício 4. Escolha do gráfico

(a) Grupo sanguíneo: nominal, gráfico de barras ordenado por frequência. Sódio sérico: contínuo,
histograma. Causas de incidentes com
medicamentos: nominal, com o objetivo de definir prioridades, gráfico de Pareto. Classe funcional:
ordinal, gráfico de barras na ordem I a IV.
(b) Quedas por mês ao longo de três anos: uma quantidade observada em ordem temporal,
gráfico de linhas com a mediana como referência.

### 4.25 Exercício 5. Cirurgias canceladas

Em ordem decrescente: sem leito 46 (38,3%), paciente sem condições clínicas 28 (acumulado 61,7%),
lista excedida 22 (80,0%), paciente não compareceu 12 (90,0%), equipamento 9
(97,5%), outros 3 (100%). As três primeiras causas respondem por 96 dos 120
cancelamentos, 80%. A disponibilidade de leitos sozinha responde por mais de um terço e
merece investigação. A escolha de uma intervenção exige também o dano causado,
a possibilidade de prevenção, a viabilidade e os recursos necessários; a frequência por si só não
estabelece a melhor primeira ação.

### 4.25 Exercício 6. Quedas mensais

Ordenar as 24 contagens destrói sua ordem no tempo, que é a informação de que uma
equipe de qualidade precisa: se as quedas aumentaram, diminuíram ou mudaram após uma intervenção.
A exibição apropriada é um gráfico de linhas das contagens em ordem de calendário, um gráfico
de sequência (run chart), com a mediana da série como linha de referência horizontal.

### 4.25 Exercício 7. Potássio

A soma é 39,9, então a média é $39{,}9/9\approx4{,}43$ mmol/L. Em ordem: 3,8; 3,9;
4,0; 4,0; 4,1; 4,2; 4,4; 4,6; 6,9; a mediana é o quinto valor, 4,1, e a
moda é 4,0. Sem o resultado hemolisado, a média é $33{,}0/8\approx4{,}12$ e a
mediana é $(4{,}0+4{,}1)/2=4{,}05$. Excluir o resultado rejeitado pelo laboratório deslocou a média em 0,31 mmol/L e
a mediana em 0,05. A exclusão segue a avaliação do laboratório, e não apenas o valor;
a hemólise, por si só, não determina o verdadeiro potássio do paciente.
Registre o motivo e obtenha uma medida válida, em vez de substituí-lo pela média.

### 4.25 Exercício 8. Uma rede de hospitais

A média ponderada é
$(1.800\times4{,}2+600\times6{,}5+200\times9{,}8)/2.600=13.420/2.600\approx5{,}16$
dias. A média simples, $(4{,}2+6{,}5+9{,}8)/3\approx6{,}83$ dias, dá ao hospital C, com
200 altas, a mesma influência do hospital A, com 1.800. Cada produto
$n_j\bar{x}_j$ corresponde aos leitos-dia de um hospital, e a média da rede é o total de
leitos-dia sobre o total de altas.

### 4.25 Exercício 9. Taxa de ocupação e leitos-dia por saída

Os leitos-dia disponíveis são $500\times31=15.500$. A taxa de ocupação é
$12.400/15.500=0{,}80$, ou 80%. Os leitos-dia por saída são $12.400/1.550=8{,}0$
dias por saída nesse mês. Essa razão do período não é necessariamente a média das
permanências completas dos 1.550 pacientes que saíram. Um paciente admitido antes do
mês contribui ao numerador apenas com os leitos-dia deste mês, enquanto um ainda
internado no fim do mês contribui com leitos-dia sem uma saída. Para usar
$n\bar{x}=\sum x$, o numerador precisa ser as permanências completas exatamente dos
pacientes contados no denominador. A taxa de ocupação e a razão do período continuam
calculadas de forma válida; suas definições operacionais respondem a perguntas diferentes.

### 4.25 Exercício 10. Quartis e cercas

Com $n=30$: a mediana tem posição $29\times0{,}5+1=15{,}5$, entre 41 e 42, então
$Md=41{,}5$ minutos. $Q_1$: $h=8{,}25$, entre 33 e 34, $Q_1=33{,}25$. $Q_3$:
$h=22{,}75$, entre 55 e 58, $Q_3=55+0{,}75\times3=57{,}25$. $IQR=24$. Cercas:
$33{,}25-36=-2{,}75$ e $57{,}25+36=93{,}25$; cerca extrema $57{,}25+72=129{,}25$. Os
candidatos são 96 e 118 minutos, ambos moderados. A haste superior termina em 83.
Os dois valores são conferidos com os registros. Se estiverem corretos, permanecem:
são os pacientes com quem o serviço mais precisa aprender.

### 4.25 Exercício 11. Uma meta como percentil

A meta diz que o percentil 90 do tempo porta-agulha deve ser de no máximo
60 minutos. $P_{90}$ tem posição $29\times0{,}9+1=27{,}1$, entre 75 e 83:
$P_{90}=75+0{,}1\times8=75{,}8$ minutos. A meta não foi cumprida: a verificação direta dá
$23/30=76{,}67\%$ tratados em até 60 minutos. Para metas operacionais, use essa
contagem como regra de decisão. Com amostras pequenas, o $P_{90}$ interpolado não precisa
dar exatamente a mesma decisão de fronteira que exigir pelo menos 90% das
observações abaixo de um ponto de corte.

### 4.25 Exercício 12. Variância à mão

A média é $780/6=130$ mmHg. Os desvios são $-12;-6;0;6;12;0$ e seus
quadrados somam $144+36+0+36+144+0=360$. Então
$s^2=360/5=72$ mmHg$^2$ e $s=\sqrt{72}\approx8{,}49$ mmHg.

### 4.25 Exercício 13. Um $\pm$ ambíguo

Se 0,3 é o erro padrão, $s=SE\times\sqrt{n}=0{,}3\times10=3{,}0$ g/dL.
Se é o desvio padrão, a dispersão é 0,3 g/dL. As duas leituras são
aritmeticamente possíveis; nem o tamanho amostral nem a média relatada resolvem
o rótulo ausente. Os autores devem identificar explicitamente a estatística, por
exemplo “média 12,2 g/dL (DP 3,0)” se 0,3 era o erro padrão, ou “média 12,2 g/dL
(DP 0,3)” se era o DP. Nenhuma afirmação sobre uma porcentagem dentro desses limites se
justifica sem uma suposição distribucional. Um intervalo de confiança de 95%
expressaria a incerteza sobre a média populacional; o DP descreve a
variação entre os pacientes observados. Relatar um não substitui o outro.

### 4.25 Exercício 14. Precisão relativa

$CV_1=100\times0{,}12/2{,}40=5{,}0\%$ e $CV_2=100\times0{,}30/7{,}50=4{,}0\%$. O método 2 tem
o maior desvio padrão e é o mais preciso em termos relativos. Os graus
Celsius formam uma escala intervalar: a conversão para Fahrenheit altera a média por
multiplicação e adição, e o desvio padrão apenas por multiplicação, de modo que o
coeficiente muda com a unidade e não descreve nenhuma propriedade dos pacientes.

### 4.25 Exercício 15. Assimetria a partir de resumos hipotéticos

A média, 42 dias, está 7 dias acima da mediana, 35. O terceiro quartil está 20 dias acima
da mediana e o primeiro, 13 dias abaixo. As duas comparações sugerem assimetria à direita.
Os autores devem começar pela mediana e pelos quartis, 35 dias [22; 55]. A média continua
relevante para os leitos-dia. Sem os dados, os resumos apenas sugerem a forma.

### 4.25 Exercício 16. Proteína C reativa

O histograma tem um pico alto abaixo de 100 mg/L e uma longa cauda à direita que chega a 982.
Média 108,7, desvio padrão 112,0, mediana 73,5, quartis 42,0 e 132,75,
assimetria ajustada 3,34; 20 valores ficam acima da cerca superior de 268,9 mg/L. O
desvio padrão grande é compatível com os valores extremos superiores; não
é um erro apenas porque excede a média. Uma
frase adequada: “A mediana da proteína C reativa na admissão foi de 73,5 mg/L
(primeiro e terceiro quartis, 42,0 e 132,8 mg/L)”.

In [18]:
# 1. Proteína C reativa: resumo, assimetria ajustada, cerca superior e valores além dela
crp = df["crp_mg_l"]
print(crp.describe().round(1))
print("Assimetria ajustada:", round(crp.skew(), 2))
q1 = crp.quantile(0.25)
q3 = crp.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("Cerca superior:", upper, "mg/L; valores além dela:", (crp > upper).sum())

count    300.0
mean     108.7
std      112.0
min        8.0
25%       42.0
50%       73.5
75%      132.8
max      982.0
Name: crp_mg_l, dtype: float64
Assimetria ajustada: 3.34
Cerca superior: 268.875 mg/L; valores além dela: 20


### 4.25 Exercício 17. Tempos de espera na triagem

Há 197 tempos de espera conhecidos. Média de 74,1 minutos, mediana 67, quartis 36 e
95, $IQR=59$, cerca superior $95+88{,}5=183{,}5$ minutos, 5 candidatos a outliers e
$P_{90}=147{,}8$ minutos. A mediana responde à pergunta sobre uma espera típica registrada:
pelo menos metade das 197 esperas observadas foi de 67 minutos ou menos. O percentil 90 responde à
pergunta com que um serviço costuma se preocupar em seguida: um paciente em dez esperou mais
do que cerca de duas horas e meia.

In [19]:
# 1. Tempos de espera do registro de triagem do Capítulo 3 (há uma cópia do arquivo em data/triage.csv), em minutos; atendimentos sem horário de atendimento ficam de fora
log = pd.read_csv("data/triage.csv", parse_dates=["arrived_at", "seen_at"])
wait = ((log["seen_at"] - log["arrived_at"]).dt.total_seconds() / 60).dropna()
print("Tempos de espera conhecidos:", len(wait), "; média:", round(wait.mean(), 1), "minutos; mediana:", wait.median(), "minutos")

# 2. Quartis, cerca superior, possíveis outliers e percentil 90
q1 = wait.quantile(0.25)
q3 = wait.quantile(0.75)
upper = q3 + 1.5 * (q3 - q1)
print("Quartis:", q1, "e", q3, "; cerca superior:", upper, "minutos; esperas além dela:", (wait > upper).sum(), "; percentil 90:", round(wait.quantile(0.9), 1), "minutos")

Tempos de espera conhecidos: 197 ; média: 74.1 minutos; mediana: 67.0 minutos
Quartis: 36.0 e 95.0 ; cerca superior: 183.5 minutos; esperas além dela: 5 ; percentil 90: 147.8 minutos


### 4.25 Exercício 18. Corrigindo uma Tabela 1

Coloque o número de incluídos no cabeçalho e identifique a população e o período
na legenda. Substitua a gravidade média por contagens e porcentagens para baixa,
moderada e alta, nessa ordem. Substitua a expressão de idade sem rótulo por
média (DP), calculada a partir das idades individuais; o erro padrão de 0,8
não é a dispersão das idades dos pacientes. Se forem os registros do capítulo, a idade
é de 68,5 anos (DP 13,8). Informe o denominador da porcentagem do sexo masculino,
166/300 = 55,3%, e relate se há valores faltantes em cada variável.
A tabela caracteriza os estudados; os critérios de elegibilidade pertencem aos
métodos do estudo.

### 4.25 Exercício 19. Acrescentando a enfermaria à Tabela 1

Clínica médica: 136 (45,3%); respiratória: 88 (29,3%); geriatria: 49 (16,3%);
terapia intensiva: 27 (9,0%). O denominador é 300 e não há valores faltantes
de enfermaria. As porcentagens somam 99,9% por causa do arredondamento. Cada contagem
resume apenas a enfermaria; nenhuma tabulação cruzada ou comparação dentro da enfermaria com
outra variável foi calculada.

In [20]:
# 1. Contagens e porcentagens das enfermarias e o número de valores faltantes de enfermaria
ward_counts = df["ward"].value_counts()
print("Pacientes por enfermaria:", ward_counts.tolist())
print("Porcentagens:", (100 * ward_counts / len(df)).round(1).tolist())
print("Valores faltantes de enfermaria:", len(df) - df["ward"].count())

Pacientes por enfermaria: [136, 88, 49, 27]
Porcentagens: [45.3, 29.3, 16.3, 9.0]
Valores faltantes de enfermaria: 0


---
*Dados sintéticos (`generate_data.py`, semente 4). Valores apenas didáticos; nada aqui é estimativa para um hospital real.*